In [9]:
import pandas as pd
from tfrecordhandler import TFRecordDataHandler
import SimpleITK as sitk
import radiomics
import config

In [10]:
batch_size = 4
tfrecord = 'input/tfrecord_dataset/full_ds.tfrecord'

In [11]:
# Initialize the data handler and radiomics feature extractor
import radiomics.featureextractor
import SimpleITK as sitk
import pandas as pd

complete_ds = TFRecordDataHandler(tfrecord, batch_size=batch_size, shuffle=False, augment=False)
feature_extractor = radiomics.featureextractor.RadiomicsFeatureExtractor(config.RADIOMICS_CONFIG_FILE)

# List to store the extracted features
all_features = []

# To store feature keys (header) for the CSV
feature_keys = None

# Iterate over the dataset, batch by batch
for images_tf, masks_tf, group_name, m_id, day_of_study in complete_ds.dataset:
    # Convert each image and mask tensor in the batch to NumPy arrays
    for image_tensor, mask_tensor, group_name_tensor, m_id_tensor, day_of_study_tensor in zip(images_tf, masks_tf, group_name, m_id, day_of_study):
        image_np = image_tensor.numpy()  # Convert TensorFlow tensor to NumPy array
        mask_np = mask_tensor.numpy()    # Convert TensorFlow tensor to NumPy array

        # Convert NumPy arrays to SimpleITK images
        image_sitk = sitk.GetImageFromArray(image_np.squeeze())  # Remove any singleton dimensions if needed
        mask_sitk = sitk.GetImageFromArray(mask_np.squeeze())

        # Handle potential errors during feature extraction
        try:
            # Extract all radiomic features
            features = feature_extractor.execute(image_sitk, mask_sitk)
            
            # Store feature keys (header) in the first iteration
            if feature_keys is None:
                feature_keys = [key for key in features.keys() if not key.startswith('diagnostics_')]
                feature_keys.insert(0, 'group_name')
                feature_keys.insert(1, 'm_id')
                feature_keys.insert(2, 'day_of_study')
            
            # Append extracted features (values) to the list, excluding diagnostics
            filtered_features = {k: v for k, v in features.items() if not k.startswith('diagnostics_')}
            all_features.append(list(filtered_features.values()))
            all_features[-1].insert(0, group_name_tensor.numpy())
            all_features[-1].insert(1, m_id_tensor.numpy())
            all_features[-1].insert(2, day_of_study_tensor.numpy())
        except Exception as e:
            # Log error details for troubleshooting
            print(f"Error extracting features for image/mask. Image shape: {image_np.shape}, Mask shape: {mask_np.shape}. Error: {str(e)}")

# Convert the list of features to a pandas DataFrame, excluding diagnostics columns
df_features = pd.DataFrame(all_features, columns=feature_keys)

# Save the features to a CSV file for machine learning
df_features.to_csv(config.FEATURES_DIR + "/radiomics_final_features__new.csv", index=False)



Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, onl

In [13]:
print("Dataset size: ", complete_ds.length )

Dataset size:  382
